In [1]:
import numpy as np
import pandas as pd

In [3]:
# reproducibility
rng = np.random.default_rng(42)

# number of patients
n = 10000

In [4]:
# disease severity
severity_levels = ["mild", "moderate", "severe"]
severity_probabilities = [0.40, 0.35, 0.25]

severity = rng.choice(
    severity_levels,
    size=n,
    p=severity_probabilities
)

In [5]:
# treatment assignment prob based on severity
treatment_probability = np.select(
    [
        severity == "mild",
        severity == "moderate",
        severity == "severe"
    ],
    [
        0.10,   # mild
        0.50,   # moderate
        0.90    # severe
    ]
)

treatment = rng.binomial(
    n=1,
    p=treatment_probability
)

In [6]:
# baseline recovery probability without treatment
baseline_recovery_probability = np.select(
    [
        severity == "mild",
        severity == "moderate",
        severity == "severe"
    ],
    [
        0.80,   # mild
        0.60,   # moderate
        0.30    # severe
    ]
)

# true causal effect:
# treatment increases recovery probability by 10%
true_treatment_effect = 0.10

recovery_probability = (
    baseline_recovery_probability + true_treatment_effect * treatment
)

In [7]:
# recovery outcome
recovery = rng.binomial(
    n=1,
    p=recovery_probability
)

In [8]:
# generate dataset
df = pd.DataFrame({
    "severity": severity,
    "treatment": treatment,
    "recovery": recovery
})

df.head(10)

,severity,treatment,recovery
0,severe,1,1
1,moderate,1,0
2,severe,1,0
3,moderate,0,1
4,mild,0,1
5,severe,1,0
6,severe,1,1
7,severe,0,0
8,mild,0,1
9,moderate,0,1


In [9]:
# check severity distribution
df["severity"].value_counts(normalize=True).sort_index()

severity
mild        0.4034
moderate    0.3524
severe      0.2442
Name: proportion, dtype: float64

In [10]:
# check treatment probability within each severity group
df.groupby("severity")["treatment"].mean()

severity
mild        0.101636
moderate    0.500568
severe      0.901720
Name: treatment, dtype: float64

In [11]:
recovery_by_group = (
    df.groupby(["severity", "treatment"])["recovery"]
      .mean()
      .unstack()
)

recovery_by_group.columns = ["untreated", "treated"]

recovery_by_group

,untreated,treated
severity,,
mild,0.803808,0.929268
moderate,0.599432,0.676304
severe,0.287500,0.400091


In [12]:
overall_recovery = df.groupby("treatment")["recovery"].mean()

overall_recovery

treatment
0    0.717817
1    0.561015
Name: recovery, dtype: float64

In [14]:
df.to_csv("confounding_synthetic_data.csv", index=False)